In [3]:
from ultralytics import YOLO
import pandas as pd
import os
import time
import torch
import numpy as np
from ultralytics.models.sam import SAM3Predictor

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [8]:
model = YOLO('/kaggle/input/datasets/roadwarrior23513245r/yolo-d-s/yolo_seg.pt')
metrics = model.val(data='/kaggle/working/data.yaml', split='test')

Ultralytics 8.4.53 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11m-seg summary (fused): 139 layers, 22,356,900 parameters, 0 gradients, 113.0 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 25.2±22.0 MB/s, size: 164.5 KB)
val: Scanning /kaggle/input/datasets/roadwarrior23513245r/warp-seg/test/labels... 520 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 520/520 145.8it/s 3.6s0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/roadwarrior23513245r/warp-seg/test is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 33/33 2.3it/s 14.5s0.4s
                   all        520       1546       0.59      0.514      0.544       0.46      0.591      0.519      0.547      0.442
           bottle-blue         87        104      0.693      0.548      0.567      0.455      0.705      0.558      0.581      0.431
          bott

In [5]:
yolo_model = YOLO("/kaggle/input/datasets/roadwarrior23513245r/samyolo/yolo12s.pt")
sam3_overrides = dict(conf=0.25, task="segment", mode="predict", model="/kaggle/input/datasets/roadwarrior23513245r/samyolo/SAM.pt")
sam3_predictor = SAM3Predictor(overrides=sam3_overrides)

def process_one_image(image_path):
    # YOLO детекция
    start_yolo = time.perf_counter()
    yolo_results = yolo_model(image_path)
    boxes = []
    for r in yolo_results:
        if r.boxes is not None:
            boxes.extend(r.boxes.xyxy.cpu().numpy().tolist())
    end_yolo = time.perf_counter()
    time_yolo = (end_yolo - start_yolo) * 1000

    # SAM3 сегментация
    sam3_predictor.set_image(image_path)
    start_sam = time.perf_counter()
    if boxes:  # если есть bbox
        _ = sam3_predictor(bboxes=boxes)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    end_sam = time.perf_counter()
    time_sam = (end_sam - start_sam) * 1000

    if hasattr(sam3_predictor, 'reset_image'):
        sam3_predictor.reset_image()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    total_time = time_yolo + time_sam
    return total_time, time_yolo, time_sam, len(boxes)


def benchmark_pipeline_from_folder(folder_path, num_warmup=2, image_extensions=('.jpg', '.jpeg', '.png')):
    image_paths = []
    for f in sorted(os.listdir(folder_path)):
        if f.lower().endswith(image_extensions):
            image_paths.append(os.path.join(folder_path, f))

    image_paths = image_paths[:20]

    if len(image_paths) == 0:
        return None, None

    # прогрев на первых num_warmup изображениях
    for i in range(min(num_warmup, len(image_paths))):
        _ = process_one_image(image_paths[i])

    # замер на оставшихся
    test_paths = image_paths[num_warmup:]
    if len(test_paths) == 0:
        return None, None

    total_times = []
    yolo_times = []
    sam_times = []

    for img_path in test_paths:
        total, t_yolo, t_sam, n_bbox = process_one_image(img_path)
        total_times.append(total)
        yolo_times.append(t_yolo)
        sam_times.append(t_sam)

    avg_total = np.mean(total_times)
    avg_yolo = np.mean(yolo_times)
    avg_sam = np.mean(sam_times)
    fps = 1000 / avg_total

    return avg_yolo, avg_sam, avg_total, fps

In [6]:
test_folder = "/kaggle/input/datasets/roadwarrior23513245r/warp-seg/test/images"
avg_yolo, avg_sam, avg_total, fps = benchmark_pipeline_from_folder(test_folder, num_warmup=2)


image 1/1 /kaggle/input/datasets/roadwarrior23513245r/warp-seg/test/images/Monitoring_photo2_04-Mar_00-47-42.jpg: 384x640 (no detections), 15.2ms
Speed: 2.1ms preprocess, 15.2ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)
Ultralytics 8.4.56 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
WARNING ⚠️ imgsz=[640] must be multiple of max stride 14, updating to [644]


/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:2156: UserWarning: Torchinductor does not support code generation for complex operators. Performance may be worse than eager.
  warnings.warn(
W0528 10:43:50.751000 57 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode



image 1/1 /kaggle/input/datasets/roadwarrior23513245r/warp-seg/test/images/Monitoring_photo_04-Mar_08-36-50.jpg: 384x640 (no detections), 14.4ms
Speed: 2.2ms preprocess, 14.4ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)
WARNING ⚠️ imgsz=[640] must be multiple of max stride 14, updating to [644]

image 1/1 /kaggle/input/datasets/roadwarrior23513245r/warp-seg/test/images/Monitoring_photo_2_test_25-Mar_11-09-46.jpg: 384x640 1 bottle, 13.7ms
Speed: 2.2ms preprocess, 13.7ms inference, 22.1ms postprocess per image at shape (1, 3, 384, 640)
WARNING ⚠️ imgsz=[640] must be multiple of max stride 14, updating to [644]

image 1/1 /kaggle/input/datasets/roadwarrior23513245r/warp-seg/test/images/Monitoring_photo_2_test_25-Mar_11-09-46.jpg: 644x644 1 0, 168.9ms
Speed: 2.5ms preprocess, 168.9ms inference, 26.6ms postprocess per image at shape (1, 3, 644, 644)
Results saved to /kaggle/working/runs/segment/predict

image 1/1 /kaggle/input/datasets/roadwarrior23513245r/warp-seg/te

In [9]:
print("\n========== РЕЗУЛЬТАТЫ ==========")
print(f"  - YOLO детекция: {avg_yolo:.2f} мс")
print(f"  - SAM3 сегментация: {avg_sam:.2f} мс")
print(f"Среднее общее время на кадр: {avg_total:.2f} мс")
print(f"Итоговый FPS (общий пайплайн): {fps:.2f}")


========== РЕЗУЛЬТАТЫ ==========
  - YOLO детекция: 15.4 мс
  - SAM3 сегментация: 60.76 мс
Среднее общее время на кадр: 76.16 мс
Итоговый FPS (общий пайплайн): 13.13
